# Sort and filter lists

**The question.** You've got a list and you want a filtered, sorted subset — "everyone who passed, highest first", "all errors from today, newest first", "the three longest words". You want the canonical Pythonic shape for this without juggling three different styles.

The canonical form is: filter with a comprehension, sort with `sorted(..., key=..., reverse=...)`. Both are one line each. Together they cover the overwhelming majority of "query this list" tasks.

In [ ]:
students = [
    {'name': 'Alice',   'grade': 85},
    {'name': 'Bob',     'grade': 42},
    {'name': 'Charlie', 'grade': 91},
    {'name': 'Diana',   'grade': 67},
    {'name': 'Eve',     'grade': 55},
]

# Filter with a list comprehension, then sort a fresh copy
passing = sorted(
    [s for s in students if s['grade'] >= 60],
    key=lambda s: s['grade'],
    reverse=True,
)

for s in passing:
    print(f"{s['name']}: {s['grade']}")


## Why it works

`sorted(iterable, key=..., reverse=...)` is the Swiss-army sort. It returns a new list (so the original stays put), takes any iterable as input (list, tuple, generator), and accepts a `key` function that is called **once per element** to produce the comparison value. That single-pass guarantee matters: even an expensive key function only runs N times, not the N·log(N) you'd get if comparisons were re-computed.

The list comprehension `[s for s in students if ...]` is Python's canonical filter. It reads as a sentence ("for each student, if they passed") and it produces a concrete list you can count, slice, or loop again. `filter(...)` exists too, but returns an iterator — fine for chaining through `sorted`, awkward when you want to use the result more than once.

Combined, the pattern is always: narrow the data (filter), then order it (sort). Doing it the other way round costs you work — you're sorting rows you're about to throw away.

## Trade-offs

**`sort()` vs `sorted()`.** `list.sort()` mutates in place and returns `None`; `sorted()` returns a new list and leaves the original alone. Use `sort()` when you own the list and don't need the original; use `sorted()` everywhere else. When in doubt, prefer `sorted` — the immutability is worth a small memory cost.

**`key=` vs `cmp=`.** Python doesn't have `cmp=` any more — `key=` is the only way to customise ordering. For multi-key sorts, return a tuple: `key=lambda s: (s['grade'], s['name'])` sorts by grade, then by name as the tiebreaker. For reversing one field but not the others, negate numeric keys (`-s['grade']`) or sort in two passes — Python's sort is **stable**, so the second pass preserves the order of the first within equal keys.

**Generators stream; lists don't.** If you're filtering a million-row log file, `(line for line in stream if error in line)` is a generator that yields one line at a time. Wrapping the same expression in `[...]` materialises the whole thing in memory. Stay lazy when the input is large and you only need the first few sorted results.

## Related reading

- [Work with higher-order functions](../../functions/recipes/work-with-higher-order-functions.ipynb) — `sorted` and `filter` are hof classics; here they are alongside `map` and `reduce`.
- [Work with nested structures](work-with-nested-structures.ipynb) — what to do once the filtered rows are themselves nested.
- [List and tuple methods](../reference/list-and-tuple-methods.md) — the full surface of list operations.
